# **FASTWOE — RECORRIDO COMPLETO DE CAPACIDADES**

Notebook de referencia sobre [**FastWoe**](https://github.com/xRiskLab/FastWoe) (xRiskLab, v0.1.8, MIT):
*encoding* Weight of Evidence + inferencia estadística, con foco en **scoring crediticio**.

Cada sección muestra una capacidad de la librería sobre un dataset crediticio público.
La comparación con la metodología propia de [`woe_example/utils.py`](../woe_example/utils.py)
está en el documento que acompaña este notebook: [`README.md`](README.md).

> **Atención al signo.** FastWoe define `WOE = ln(P(evento|bin) / P(no evento|bin)) − ln(P(evento) / P(no evento))`,
> con **evento = malo**. Tu `utils.py` usa `ln(%buenos / %malos)`. Son **convenciones opuestas**:
> acá el bin más riesgoso tiene **WOE positivo**. El detalle está en el `README.md`.

---
## **0. INSTALACIÓN EN EL KERNEL SELECCIONADO**

Usá la *magic* `%pip`, **no** `!pip`.

- `%pip` instala en el intérprete del kernel que tenés seleccionado arriba a la derecha.
- `!pip` va al `pip` que esté primero en el `PATH` del shell, que muchas veces **no** es el del kernel.
  Esa es la causa habitual del "lo instalé pero el `import` sigue fallando".

Si estás detrás de un proxy corporativo con inspección SSL (típico en Telecom) y ves
`SSLCertVerificationError`, descomentá la segunda celda: agrega los *trusted hosts* de PyPI.

In [ ]:
# Instalación estándar. El extra [plotting] trae matplotlib, necesario para las secciones 12 y 13.
%pip install "fastwoe[plotting]"

# Extras opcionales:
#   %pip install "fastwoe[faiss]"      # binning por KMeans (FAISS, CPU)
#   %pip install "fastwoe[faiss-gpu]"  # idem, con CUDA 12
#   %pip install "fastwoe[examples]"   # seaborn, jupyter, statsmodels, pygam

In [ ]:
# Solo si la celda anterior falla con SSLCertVerificationError (proxy con inspección SSL).
# %pip install --trusted-host pypi.org --trusted-host files.pythonhosted.org "fastwoe[plotting]"

### Verificación de que la instalación cayó en el kernel correcto

`sys.executable` tiene que apuntar al Python del kernel seleccionado. Si `import fastwoe`
funciona acá, la instalación quedó bien.

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
import fastwoe

print("Kernel  :", sys.executable)
print("Python  :", sys.version.split()[0])
print("fastwoe :", fastwoe.__version__)
print("sklearn :", sklearn.__version__, " (FastWoe requiere >=1.3.0, <1.8.0)")
print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)

# Los 11 nombres públicos de la librería:
print("\n__all__ :", fastwoe.__all__)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

RANDOM_STATE = 42

---
## **1. DATOS: GERMAN CREDIT (`credit-g`)**

Dataset crediticio público de OpenML: 1.000 solicitudes, 20 variables (13 categóricas + 7 numéricas),
target `good` / `bad`. Se elige porque **mezcla categóricas y numéricas** y es del dominio correcto.

Dos detalles importantes:

1. **El target se mapea a `1 = bad`**, es decir *evento = default*. Queda alineado con la convención
   `malos = target == 1` de tu `utils.py`, aunque el signo del WOE resultante sea el opuesto.
2. Si no hay red (o el proxy bloquea OpenML), la celda **cae automáticamente a un dataset sintético**
   con el mismo esquema de columnas, para que el notebook corra igual de punta a punta.

> **Trampa a evitar: el dtype `category`.** OpenML devuelve las columnas categóricas con dtype
> `category` de pandas. En fastwoe 0.1.8, `predict_ci()` escribe valores *float* sobre las columnas
> de entrada, y pandas rechaza eso en una columna `category` con
> `TypeError: Cannot setitem on a Categorical with a new category`.
> `fit`, `transform` y `predict_proba` no se ven afectados — **solo rompe `predict_ci`**.
> Por eso el *loader* convierte `category` → `str`. Si traés datos de BigQuery con
> `dtype="category"`, te va a pasar lo mismo.

In [ ]:
from sklearn.datasets import fetch_openml

def _normalizar_dtypes(df):
    '''
    Pasa las columnas 'category' de pandas a str.

    IMPORTANTE: no es cosmetico. En fastwoe 0.1.8, predict_ci() y
    WeightOfEvidence.predict_ci() escriben valores float sobre las columnas
    de entrada; si el dtype es 'category' pandas rechaza la asignacion con
    "Cannot setitem on a Categorical with a new category". credit-g viene de
    OpenML con dtype 'category', asi que sin esta conversion las secciones 8
    y 11 fallan. fit/transform/predict_proba andan igual: solo rompe predict_ci.
    '''
    cats = df.select_dtypes(include=["category"]).columns
    for c in cats:
        df[c] = df[c].astype(str)
    return df


def cargar_credit_g():
    '''Devuelve (X, y, origen). Cae a sintetico si no hay red.'''
    try:
        bunch = fetch_openml("credit-g", version=1, as_frame=True, parser="auto")
        df = bunch.frame.copy()
        # class: 'good' / 'bad'  ->  target: 0 / 1  (evento = bad = default)
        y = (df.pop("class") == "bad").astype(int).rename("target")
        return _normalizar_dtypes(df), y, "OpenML credit-g"
    except Exception as e:
        print(f"[aviso] No se pudo bajar credit-g ({type(e).__name__}). Uso dataset sintético.\n")
        return _credit_g_sintetico()


def _credit_g_sintetico(n=1000, seed=RANDOM_STATE):
    '''Replica sintetica con las mismas columnas y relaciones de riesgo conocidas.'''
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "duration":               rng.integers(4, 72, n),
        "credit_amount":          rng.lognormal(7.8, 0.6, n).round(),
        "age":                    rng.integers(19, 76, n),
        "installment_commitment": rng.integers(1, 5, n),
        "residence_since":        rng.integers(1, 5, n),
        "existing_credits":       rng.integers(1, 5, n),
        "num_dependents":         rng.integers(1, 3, n),
        "checking_status":        rng.choice(["<0", "0<=X<200", ">=200", "no checking"], n),
        "credit_history":         rng.choice(["critical/other existing credit", "existing paid",
                                              "delayed previously", "all paid", "no credits/all paid"], n),
        "purpose":                rng.choice(["radio/tv", "new car", "used car", "furniture/equipment",
                                              "business", "education", "repairs"], n),
        "savings_status":         rng.choice(["<100", "100<=X<500", "500<=X<1000", ">=1000", "no known savings"], n),
        "employment":             rng.choice(["unemployed", "<1", "1<=X<4", "4<=X<7", ">=7"], n),
        "personal_status":        rng.choice(["male single", "female div/dep/mar", "male div/sep"], n),
        "property_magnitude":     rng.choice(["real estate", "life insurance", "car", "no known property"], n),
        "housing":                rng.choice(["own", "rent", "for free"], n),
        "job":                    rng.choice(["skilled", "unskilled resident", "high qualif/self emp/mgmt"], n),
        "other_parties":          rng.choice(["none", "guarantor", "co applicant"], n),
        "other_payment_plans":    rng.choice(["none", "bank", "stores"], n),
        "own_telephone":          rng.choice(["yes", "none"], n),
        "foreign_worker":         rng.choice(["yes", "no"], n),
    })
    # Riesgo: plazo largo sube, edad alta baja, monto alto sube, sin cuenta corriente baja.
    logit = (-1.2
             + 0.030 * df["duration"]
             - 0.020 * (df["age"] - 35)
             + 0.00008 * df["credit_amount"]
             - 0.60 * (df["checking_status"] == "no checking")
             + 0.40 * (df["purpose"] == "business"))
    y = pd.Series(rng.binomial(1, 1 / (1 + np.exp(-logit))), name="target")
    return _normalizar_dtypes(df), y, "sintético (fallback sin red)"


X, y, ORIGEN = cargar_credit_g()
print(f"Origen : {ORIGEN}")
print(f"Shape  : {X.shape}")
print(f"Target : {y.value_counts().to_dict()}  ->  tasa de malos = {y.mean():.1%}")
X.head()

### Columna de alta cardinalidad

`credit-g` no trae ninguna variable con cardinalidad realmente alta (el máximo son ~10 niveles).
Para que la sección de `WoePreprocessor` tenga algo real que reducir, cruzamos dos categóricas
—`purpose` × `credit_history`—, que es *feature engineering* legítimo y no un dato inventado.

In [ ]:
X["purpose_x_history"] = X["purpose"].astype(str) + " | " + X["credit_history"].astype(str)
print("Niveles de purpose_x_history:", X["purpose_x_history"].nunique())
X["purpose_x_history"].value_counts().head(8)

### Partición train / test

El *split* no es decorativo: varias secciones muestran `fit` en **train** y `transform` en **test**.
Ese es justamente el mecanismo con el que FastWoe **congela los cortes** del binning, el equivalente
a tu `percentiles_precalculados` en `discretizar_variables`.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

NUM = X.select_dtypes(include=np.number).columns.tolist()
CAT = [c for c in X.columns if c not in NUM]

print(f"train {X_tr.shape} | test {X_te.shape}")
print(f"\nNuméricas ({len(NUM)}): {NUM}")
print(f"\nCategóricas ({len(CAT)}): {CAT}")

---
## **2. QUICK START: `fit_transform` Y `get_mapping`**

El camino mínimo. `FastWoe` es un *transformer* de scikit-learn: `fit`, `transform`, `fit_transform`.

Detecta sola qué columna es categórica y cuál numérica: toda numérica con **más de
`numerical_threshold` valores únicos** (default 20) se bincea antes de calcular el WOE.

In [ ]:
from fastwoe import FastWoe

woe = FastWoe(random_state=RANDOM_STATE)
X_tr_woe = woe.fit_transform(X_tr, y_tr)

print("Antes :", X_tr.shape, "->  Después:", X_tr_woe.shape, " (mismo shape, valores reemplazados por WOE)")
X_tr_woe.head()

`get_mapping(feature)` es el equivalente directo de una hoja de tu Excel de bivariados,
**pero con inferencia estadística incluida**.

Columnas que devuelve:

| Columna | Qué es |
|---|---|
| `category` | la categoría o el intervalo del bin |
| `count`, `count_pct` | volumen y participación del bin |
| `good_count`, `bad_count` | buenos y malos del bin |
| `event_rate` | tasa de malos del bin — tu `porcmalogrupo` |
| `woe` | Weight of Evidence (signo opuesto al tuyo) |
| `woe_se` | **error estándar del WOE** — esto no existe en `utils.py` |
| `woe_ci_lower`, `woe_ci_upper` | intervalo de confianza al 95 % |

In [ ]:
woe.get_mapping("checking_status")

In [ ]:
# Para una numérica, las categorías son los intervalos que encontró el binning.
woe.get_mapping("duration")

> **Cómo leer el `woe_se`.** Un bin con `woe_ci_lower` y `woe_ci_upper` de signos distintos
> tiene un WOE **no distinguible de cero** al 95 %: ese bin no está aportando evidencia,
> por más que su WOE puntual se vea grande. En tu Excel actual esa distinción no se puede hacer,
> y es la razón principal por la que bins chicos parecen informativos cuando no lo son.

---
## **3. ALTA CARDINALIDAD: `WoePreprocessor`**

Antes de calcular WOE conviene recortar las categorías raras: un nivel con 3 casos produce
un WOE enorme y completamente inestable. `WoePreprocessor` agrupa la cola en un token `__other__`.

Tres criterios, combinables:

| Parámetro | Efecto |
|---|---|
| `max_categories` | se queda con las N más frecuentes |
| `top_p` | se queda con las que cubren el `top_p` de la masa (default 0.95) |
| `min_count` | descarta las que tienen menos de N casos (default 10) |
| `other_token` | etiqueta del grupo residual (default `"__other__"`) |

In [ ]:
from fastwoe import WoePreprocessor

prep = WoePreprocessor(top_p=0.95, min_count=15)

# cat_features permite aplicar la reducción SOLO a las columnas que te interesan.
prep.fit(X_tr[CAT], cat_features=["purpose_x_history"])

resumen = prep.get_reduction_summary(X_tr[CAT])
resumen

In [ ]:
X_tr_prep = prep.transform(X_tr[CAT])

antes  = X_tr["purpose_x_history"].nunique()
despues = X_tr_prep["purpose_x_history"].nunique()
print(f"purpose_x_history: {antes} niveles  ->  {despues} niveles")

X_tr_prep["purpose_x_history"].value_counts().tail(5)

In [ ]:
# get_category_mapping() devuelve, por columna, las categorías que se conservan.
# Ojo: los valores son SETS de Python, no listas -> no se pueden indexar directamente.
mapa = prep.get_category_mapping()

pd.Series(
    {col: f"{len(cats)} conservadas: {sorted(cats)[:3]}..." for col, cats in mapa.items()},
    name="categorias_conservadas",
).to_frame()

> **Ojo con el orden.** `WoePreprocessor` se fitea en **train** y se aplica a test. Una categoría
> que aparece solo en test cae automáticamente en `__other__`, que ya tiene un WOE estimado.
> Es el manejo de categorías no vistas que hoy en `aplicar_woe` te dejaría un `NaN`.

---
## **4. BINNING DE VARIABLES NUMÉRICAS**

Acá está la diferencia conceptual más grande con `discretizar_variables`: FastWoe bincea
**de forma supervisada** (mirando el target), no por percentiles ciegos.

| `binning_method` | Cómo corta | Cuándo usarlo |
|---|---|---|
| `"tree"` *(default)* | árbol de decisión: los cortes maximizan separación del target | scoring crediticio |
| `"kbins"` | `KBinsDiscretizer` de sklearn: `quantile`, `uniform` o `kmeans` | **el más parecido a tu `metodo='quantile'`** |
| `"faiss_kmeans"` | clustering KMeans con FAISS (CPU/GPU) | volúmenes muy grandes |

`numerical_threshold` (default 20) define a partir de cuántos valores únicos una numérica se bincea.

In [ ]:
# --- A) Binning por árbol (supervisado, default)
woe_tree = FastWoe(
    binning_method="tree",
    tree_kwargs={"max_depth": 3, "min_samples_leaf": 40},
    random_state=RANDOM_STATE,
)
woe_tree.fit(X_tr, y_tr)

woe_tree.get_binning_summary()

In [ ]:
# --- B) Binning por cuantiles: el equivalente a tu metodo='quantile' con n_bins=5
woe_kbins = FastWoe(
    binning_method="kbins",
    binner_kwargs={"n_bins": 5, "strategy": "quantile", "encode": "ordinal"},
    random_state=RANDOM_STATE,
)
woe_kbins.fit(X_tr, y_tr)

print("Cortes por cuantiles (equivalente a pd.qcut(q=5)):")
woe_kbins.get_mapping("duration")

In [ ]:
# --- Comparación directa de los dos criterios sobre la misma variable
comp = pd.concat([
    woe_tree.get_mapping("duration").assign(metodo="tree")[["metodo", "category", "count", "event_rate", "woe"]],
    woe_kbins.get_mapping("duration").assign(metodo="kbins")[["metodo", "category", "count", "event_rate", "woe"]],
], ignore_index=True)
comp

Con `binning_method="tree"` podés además inspeccionar el árbol que produjo los cortes:

In [ ]:
# Los puntos de corte exactos, listos para traducir a un CASE WHEN de SQL.
cortes = woe_tree.get_split_value_histogram("duration", as_array=True)
print("duration      :", cortes)
print("credit_amount :", woe_tree.get_split_value_histogram("credit_amount"))

# Y el estimador sklearn subyacente, por si querés graficarlo o auditarlo.
arbol = woe_tree.get_tree_estimator("duration")
print("\nEstimador:", type(arbol).__name__, "| profundidad:", arbol.get_depth(), "| hojas:", arbol.get_n_leaves())

In [ ]:
# --- C) FAISS KMeans (requiere el extra [faiss]; se saltea si no está instalado)
try:
    woe_faiss = FastWoe(
        binning_method="faiss_kmeans",
        faiss_kwargs={"k": 5, "niter": 20, "verbose": False, "gpu": False},
        random_state=RANDOM_STATE,
    )
    woe_faiss.fit(X_tr[NUM], y_tr)
    display(woe_faiss.get_binning_summary())
except ImportError:
    print("FAISS no instalado. Para probarlo:  %pip install \"fastwoe[faiss]\"")
except Exception as e:
    print(f"FAISS no disponible ({type(e).__name__}): {e}")

> **Qué te sirve de acá.** `get_split_value_histogram()` te devuelve los cortes como array.
> Eso es exactamente lo que necesitás para escribir el `CASE WHEN` en BigQuery: podés dejar que
> el árbol elija los cortes en Python y después **materializarlos como constantes en SQL**,
> sin depender de `PERCENTILE_CONT` en tiempo de ejecución.

---
## **5. MONOTONICIDAD FORZADA (`monotonic_cst`)**

Requisito duro de cualquier *scorecard* que vaya a validación: el riesgo tiene que moverse en una
sola dirección a lo largo de los bins. Hoy vos verificás esto **a ojo**, con la escala de color
verde-amarillo-rojo del Excel. FastWoe lo **impone en el binning**.

- `1` → el riesgo **crece** con la variable
- `-1` → el riesgo **decrece**
- `0` → sin restricción

**Solo funciona con `binning_method="tree"`.**

In [ ]:
woe_mono = FastWoe(
    binning_method="tree",
    monotonic_cst={
        "duration": 1,       # más plazo  -> más riesgo
        "age": -1,           # más edad   -> menos riesgo
        "credit_amount": 1,  # más monto  -> más riesgo
    },
    numerical_threshold=10,
    random_state=RANDOM_STATE,
)
woe_mono.fit(X_tr, y_tr)

woe_mono.get_binning_summary()[["feature", "n_bins", "method", "monotonic_constraint"]]

In [ ]:
# Verificación: el WOE ahora es monótono creciente a lo largo de los bins de duration.
m = woe_mono.get_mapping("duration")
m[["category", "count", "event_rate", "woe", "woe_ci_lower", "woe_ci_upper"]]

In [ ]:
# Sin restricción vs con restricción, lado a lado.
sin_r = woe_tree.get_mapping("age")[["category", "count", "event_rate", "woe"]]
con_r = woe_mono.get_mapping("age")[["category", "count", "event_rate", "woe"]]

print("--- age SIN restricción ---");  display(sin_r)
print("--- age CON monotonic_cst=-1 ---"); display(con_r)

def es_monotona(serie):
    d = serie.diff().dropna()
    return "creciente" if (d >= 0).all() else ("decreciente" if (d <= 0).all() else "NO monótona")

print("WOE sin restricción :", es_monotona(sin_r["woe"]))
print("WOE con restricción :", es_monotona(con_r["woe"]))

---
## **6. INFERENCIA ESTADÍSTICA: IV CON ERROR ESTÁNDAR Y SIGNIFICANCIA**

Esta es la sección que más se aleja de `utils.py`. Tu `iv_total_variable` es un número puntual
que ordena variables; acá el IV viene con **error estándar, intervalo de confianza y un veredicto
de significancia**, más el Gini y el Somers' D de cada variable.

In [ ]:
# Estadísticas completas por variable.
stats = woe.get_feature_stats()
stats

In [ ]:
# Versión compacta, ordenada por poder predictivo. Equivale a tu ranking por iv_total_variable.
woe.get_feature_summary()

In [ ]:
# El análisis que no podés hacer hoy: ¿este IV es distinguible de cero?
iv = woe.get_iv_analysis(alpha=0.05)
iv

In [ ]:
# Lectura contra los umbrales clásicos de Siddiqi.
def fuerza_iv(v):
    if v < 0.02:  return "inútil"
    if v < 0.10:  return "débil"
    if v < 0.30:  return "media"
    if v < 0.50:  return "fuerte"
    return "sospechosa (revisar leakage)"

tabla = iv[["feature", "iv", "iv_ci_lower", "iv_ci_upper", "iv_significance"]].copy()
tabla["fuerza"] = tabla["iv"].map(fuerza_iv)
tabla

> **El punto importante.** Una variable puede tener IV = 0,04 (*"débil pero la dejo"*) y a la vez
> `iv_significance = "Not Significant"`, es decir un IV **estadísticamente indistinguible de cero**.
> Ese es ruido que se te cuela en el modelo. Con tu IV puntual —y encima en escala ×100— esa
> variable se ve igual que una genuinamente débil pero real.

In [ ]:
# También podés pedir el detalle de una sola variable.
woe.get_iv_analysis("purpose")

In [ ]:
# get_probability_mapping: en vez del WOE, la probabilidad de evento por categoría.
woe.get_probability_mapping("checking_status")

In [ ]:
# get_all_mappings: el diccionario completo {variable: DataFrame}.
# Es el equivalente de tu calcular_woe_mapping(), pero ya calculado dentro del fit.
todos = woe.get_all_mappings()
print(f"{len(todos)} variables mapeadas")
print(list(todos.keys()))

---
## **7. MODOS DE SALIDA DE `transform`**

`transform(X, output=...)` no devuelve solo el WOE. Los cinco modos:

| `output` | Qué devuelve |
|---|---|
| `"woe"` *(default)* | el WOE del bin |
| `"woe_norm"` | WOE / SE — WOE **normalizado por su incertidumbre** |
| `"wald"` | estadístico de Wald: `(WOE + log-odds prior) / SE` |
| `"woe_upper_ci"` | cota superior del IC 95 % |
| `"woe_lower_ci"` | cota inferior del IC 95 % |

In [ ]:
modos = ["woe", "woe_norm", "wald", "woe_upper_ci", "woe_lower_ci"]
salidas = {m: woe.transform(X_te, output=m)["duration"] for m in modos}

pd.DataFrame(salidas, index=X_te.index).head(10)

`"woe_norm"` es el modo con más valor práctico: penaliza los bins con pocos casos. Un WOE de 1,5
estimado sobre 12 observaciones tiene un SE grande, así que su WOE normalizado queda chico.
Es una forma de *shrinkage* implícito que hoy no tenés.

In [ ]:
# Efecto del shrinkage: bins chicos pierden peso al normalizar.
m = woe.get_mapping("purpose_x_history").copy()
m["woe_norm"] = m["woe"] / m["woe_se"]
m[["category", "count", "woe", "woe_se", "woe_norm"]].sort_values("count").head(10)

In [ ]:
# transform_standardized: versión estandarizada, útil para comparar magnitudes entre variables.
woe.transform_standardized(X_te).head()

---
## **8. PREDICCIÓN E INTERVALOS DE CONFIANZA**

`FastWoe` no es solo un *encoder*: sumando los WOE de todos los bins de un caso obtiene un score
y, vía Naive Bayes, una probabilidad. Sirve como **baseline instantáneo** contra el cual medir
cualquier modelo posterior.

In [ ]:
proba = woe.predict_proba(X_te)          # (n, 2) -> columna 1 = P(malo)
pred  = woe.predict(X_te)                # clase predicha
ci    = woe.predict_ci(X_te, alpha=0.05) # ndarray (n, 2) -> [inferior, superior]

print("predict_proba:", proba.shape, "| predict_ci:", type(ci).__name__, ci.shape)

resultado = pd.DataFrame({
    "p_malo":   proba[:, 1],
    "ci_lower": ci[:, 0],
    "ci_upper": ci[:, 1],
    "pred":     pred,
    "real":     y_te.values,
}, index=X_te.index)
resultado.head(10)

> **Corrección al README oficial de FastWoe.** La documentación del repo muestra
> `ci_results[['prediction', 'lower_ci', 'upper_ci']]`, como si `predict_ci` devolviera un DataFrame.
> En la versión 0.1.8 devuelve un **`np.ndarray` de shape (n, 2)**. Verificado contra la librería instalada.

In [ ]:
from sklearn.metrics import roc_auc_score

auc  = roc_auc_score(y_te, proba[:, 1])
gini = 2 * auc - 1
print(f"Baseline WOE + Naive Bayes  ->  AUC = {auc:.4f} | Gini = {gini:.4f}")

# Ancho del intervalo: cuánta incertidumbre arrastra cada predicción.
ancho = ci[:, 1] - ci[:, 0]
print(f"\nAncho del IC 95%: mediana = {np.median(ancho):.4f} | máximo = {ancho.max():.4f}")
print("Los casos con IC más ancho son los que caen en bins con poca población.")

---
## **9. WOE PIECEWISE**

El *trade-off* clásico al pasar a regresión logística:

- **WOE clásico**: 1 columna por variable → 1 solo coeficiente → asume que el WOE ya captura toda la forma.
- **Dummies**: 1 columna por bin → máxima flexibilidad → explosión de parámetros y sobreajuste.

El **piecewise WOE** (Raymond Anderson, Standard Bank) es el punto medio: agrupa los bins en
*pieces* y le da a la logística **un coeficiente por pieza**.

`assign_pieces(strategy="sign")` parte por el signo del WOE: bins con WOE < 0 en la pieza 0,
bins con WOE ≥ 0 en la pieza 1.

In [ ]:
woe_pw = FastWoe(random_state=RANDOM_STATE)
woe_pw.fit(X_tr, y_tr)
woe_pw.assign_pieces(strategy="sign")

# La asignación queda en mappings_, como una columna extra 'piece'.
woe_pw.mappings_["duration"][["count", "event_rate", "woe", "piece"]]

In [ ]:
X_tr_pw = woe_pw.transform(X_tr, output="piecewise")
X_te_pw = woe_pw.transform(X_te, output="piecewise")

print(f"WOE clásico : {X_tr_woe.shape[1]} columnas")
print(f"Piecewise   : {X_tr_pw.shape[1]} columnas")
print(f"\nNomenclatura: {list(X_tr_pw.columns)[:6]}")
X_tr_pw.head()

In [ ]:
from sklearn.linear_model import LogisticRegression

# penalty=None: queremos los coeficientes crudos, sin regularización.
lr_pw = LogisticRegression(penalty=None, max_iter=1000).fit(X_tr_pw, y_tr)
lr_cl = LogisticRegression(penalty=None, max_iter=1000).fit(woe_pw.transform(X_tr), y_tr)

auc_pw = roc_auc_score(y_te, lr_pw.predict_proba(X_te_pw)[:, 1])
auc_cl = roc_auc_score(y_te, lr_cl.predict_proba(woe_pw.transform(X_te))[:, 1])

print(f"Logística sobre WOE clásico  : AUC test = {auc_cl:.4f} | Gini = {2*auc_cl-1:.4f}")
print(f"Logística sobre WOE piecewise: AUC test = {auc_pw:.4f} | Gini = {2*auc_pw-1:.4f}")

In [ ]:
# Coeficientes por pieza: en WOE clásico bien construido deberían dar cerca de 1.
coefs = pd.DataFrame({"columna": X_tr_pw.columns, "coef": lr_pw.coef_[0]})
coefs.sort_values("coef", key=np.abs, ascending=False).head(12)

In [ ]:
# También podés definir las piezas a mano, con piece_map.
cats_ch = list(woe_pw.mappings_["checking_status"].index)
print("Categorías de checking_status:", cats_ch)

woe_pw.assign_pieces(piece_map={
    "checking_status": {c: (0 if i < len(cats_ch) // 2 else 1) for i, c in enumerate(cats_ch)}
})
woe_pw.mappings_["checking_status"][["count", "woe", "piece"]]

> **Por qué esto importa para tu repo.** Tus resúmenes marcan que el paso de WOE a regresión
> logística no está implementado en `woe_example`. El piecewise WOE es exactamente ese puente,
> y es el tema central del capítulo 2 del libro
> ([`02_logistic_woe_cap2.md`](../resumenes/02_logistic_woe_cap2.md)).

---
## **10. INTEGRACIÓN CON `Pipeline` DE SCIKIT-LEARN**

`WoePreprocessor` y `FastWoe` son transformers de sklearn en regla, así que entran en un `Pipeline`
y en `cross_val_score` sin adaptadores. Esto elimina de raíz el *leakage* de calcular el WOE sobre
todo el dataset antes de partir: dentro del pipeline, el WOE se re-estima en cada *fold*.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

pipe = Pipeline([
    ("prep", WoePreprocessor(top_p=0.95, min_count=15)),
    ("woe",  FastWoe(binning_method="tree", random_state=RANDOM_STATE)),
    ("clf",  LogisticRegression(max_iter=1000)),
])

pipe.fit(X_tr, y_tr)
auc_pipe = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
print(f"Pipeline completo -> AUC test = {auc_pipe:.4f} | Gini = {2*auc_pipe-1:.4f}")

In [ ]:
# Validación cruzada SIN leakage: el WOE se recalcula dentro de cada fold.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="roc_auc")

print(f"AUC por fold : {np.round(scores, 4)}")
print(f"AUC media    : {scores.mean():.4f} (+/- {scores.std():.4f})")

> **Contraste con tu flujo actual.** Hoy `calcular_bivariados` corre sobre el dataframe entero
> traído de BigQuery. Si después partís train/test, el WOE ya vio los datos de test: el *leakage*
> es estructural y te infla la performance medida. El `Pipeline` lo resuelve gratis.

---
## **11. EXPLICABILIDAD: `WeightOfEvidence`**

Explicación **caso a caso**: cuánto aportó cada variable a la decisión sobre un solicitante puntual.
En crédito esto no es opcional — es lo que se necesita para fundamentar un rechazo.

Dos detalles del API que la documentación oficial no deja claros:

1. El `classifier` tiene que ser un **`FastWoe`**, no un `LogisticRegression`. Si le pasás un
   estimador de sklearn tira `ValueError: Only FastWoe classifiers are supported`.
2. Si no le pasás `classifier`, lo **crea y fitea solo** a partir de `X_train` / `y_train`.
3. Recibe los datos **originales**, no los ya transformados a WOE.

In [ ]:
from fastwoe import WeightOfEvidence

# Variante A: reusar un FastWoe ya fiteado.
explainer = WeightOfEvidence(
    classifier=woe,
    X_train=X_tr,
    y_train=y_tr,
    class_names=["Bueno", "Malo"],
)

# Variante B: dejar que lo cree solo.
# explainer = WeightOfEvidence(X_train=X_tr, y_train=y_tr)

print(explainer.summary())

In [ ]:
# Explicación de un caso individual.
expl = explainer.explain(X_te, sample_idx=0, true_labels=y_te)

print(f"Predicción      : {expl['predicted_label']}")
print(f"Probabilidades  : {expl['predicted_proba']}")
print(f"WOE total       : {expl['total_woe']:.4f}")
print(f"Interpretación  : {expl['interpretation']}")
print("\nAporte de cada variable:")

aportes = (pd.Series(expl["feature_contributions"], name="woe_aporte")
             .sort_values(key=np.abs, ascending=False)
             .to_frame())
aportes

In [ ]:
# Misma explicación, con intervalos de confianza sobre el WOE total.
expl_ci = explainer.explain_ci(X_te, sample_idx=0, alpha=0.05)

print(f"Nivel de confianza : {expl_ci['confidence_level']}")
print(f"WOE total          : {expl_ci['total_woe']:.4f}")
print(f"Escenario conservador : {expl_ci['ci_conservative']}")
print(f"Escenario optimista   : {expl_ci['ci_optimistic']}")
print(f"Rango de incertidumbre: {expl_ci['uncertainty_range']}")

In [ ]:
# predict_ci del explainer: predicciones con banda de incertidumbre para un lote.
lote = explainer.predict_ci(X_te.head(50), alpha=0.05, return_probabilities=True)

print("Claves:", list(lote.keys()))
print("\nResumen de incertidumbre:")
lote["uncertainty_summary"]

---
## **12. VISUALIZACIÓN**

Dos funciones, ambas dependen del extra `[plotting]`.

- `visualize_woe(...)` → curva de WOE por bin, o *waterfall* de una predicción individual.
- `plot_performance(...)` → curva CAP con el Gini calculado.

In [ ]:
import matplotlib.pyplot as plt
from fastwoe import visualize_woe

# Curva de WOE por bin. mode="proba" muestra tasas; mode="logit" muestra WOE en escala log-odds.
# Devuelve ademas el DataFrame con los datos que grafico.
datos_plot = visualize_woe(woe, feature_name="duration", mode="proba", figsize=(10, 5))
plt.show()
datos_plot

In [ ]:
# Escala logit: acá se ve directamente la monotonicidad del WOE.
visualize_woe(woe_mono, feature_name="duration", mode="logit", figsize=(10, 5))
plt.show()

In [ ]:
# Waterfall: cómo se arma la decisión de UN caso, variable por variable.
visualize_woe(woe, explanation=expl, figsize=(10, 5))
plt.show()

In [ ]:
from fastwoe import plot_performance

# Curva CAP. Devuelve (figura, ejes, gini).
fig, ax, gini_cap = plot_performance(
    y_te.values, proba[:, 1],
    labels=["WOE + Naive Bayes"], figsize=(7, 5),
)
plt.show()
print(f"Gini = {gini_cap}")

In [ ]:
# Comparar varios modelos en la misma curva: se pasa una lista de scores.
fig, ax, ginis = plot_performance(
    y_te.values,
    [proba[:, 1], pipe.predict_proba(X_te)[:, 1], lr_pw.predict_proba(X_te_pw)[:, 1]],
    labels=["Naive Bayes WOE", "Pipeline logística", "Logística piecewise"],
    figsize=(7, 5),
)
plt.show()
print("Ginis:", ginis)

---
## **13. DISPLAY ENRIQUECIDO**

El módulo `display` formatea las tablas de WOE e IV directamente en el notebook, con escala de
color incluida. Es el reemplazo natural de tu `exportar_multiples_bivariados_excel`: mismo objetivo
—leer monotonicidad de un vistazo— sin salir de Python ni depender de `xlsxwriter`.

In [ ]:
from fastwoe import style_woe_mapping, style_iv_analysis, StyledDataFrame

# Mapeo de una variable, con formato.
style_woe_mapping(woe.get_mapping("duration"), feature_name="duration", theme="light")

In [ ]:
# Análisis de IV de todas las variables, con formato y semáforo de significancia.
style_iv_analysis(woe.get_iv_analysis(), theme="light")

In [ ]:
# Los decoradores styled / iv_styled permiten formatear la salida de funciones propias.
from fastwoe import styled

@styled(title="Bivariado propio", subtitle="Formateado con el decorador de FastWoe", precision=4)
def mi_bivariado(encoder, variable):
    return encoder.get_mapping(variable)[["category", "count", "event_rate", "woe", "woe_se"]]

mi_bivariado(woe, "checking_status")

---
## **14. MÉTRICAS Y SELECCIÓN DE VARIABLES**

FastWoe trae un módulo de métricas basado en **Somers' D** calculado con Numba, y un algoritmo
de selección *stepwise* que es una alternativa seria al ranking por IV.

In [ ]:
from fastwoe import gini_contributions

contribs, gini_total = gini_contributions(proba[:, 1], y_te.values)

print(f"Gini total                    : {gini_total:.6f}")
print(f"Suma de las contribuciones    : {contribs.sum():.6f}   <- coinciden por construcción")

# Qué casos individuales están destruyendo el poder discriminante del score.
peores = pd.DataFrame({
    "contribucion": contribs,
    "p_malo": proba[:, 1],
    "real": y_te.values,
}, index=X_te.index).sort_values("contribucion")

print("\nLos 8 casos que más restan al Gini:")
peores.head(8)

### Selección *stepwise* por Somers' D marginal

`marginal_somersd_selection` es conceptualmente distinto de ordenar por IV: en cada paso ajusta un
modelo con lo ya seleccionado, calcula los **residuos**, y elige la variable que mejor correlaciona
con lo que **todavía no está explicado**. Penaliza la redundancia de forma natural — algo que el
ranking por IV, que es puramente univariado, no hace.

In [ ]:
from fastwoe import screening

seleccion = screening.marginal_somersd_selection(
    X_tr, y_tr,
    X_test=X_te, y_test=y_te,
    min_msd=0.01,
    correlation_threshold=0.5,
    random_state=RANDOM_STATE,
    verbose=False,
)

print("Claves:", list(seleccion.keys()))
print("\nVariables seleccionadas, en orden de entrada:")
for i, f in enumerate(seleccion["selected_features"], 1):
    print(f"  {i}. {f}")

In [ ]:
# Somers' D univariado vs. el aporte marginal en cada paso.
print("--- Somers' D univariado ---")
display(pd.Series(seleccion["univariate_somersd"], name="somersd").sort_values(ascending=False))

print("--- Aporte marginal por paso (msd_history) ---")
display(pd.DataFrame(seleccion["msd_history"]))

In [ ]:
# Contraste: el orden por IV NO es el mismo que el orden por aporte marginal.
orden_iv  = woe.get_feature_summary()["feature"].tolist()
orden_msd = seleccion["selected_features"]

pd.DataFrame({
    "ranking": range(1, min(len(orden_iv), len(orden_msd)) + 1),
    "por IV (univariado)": orden_iv[:len(orden_msd)],
    "por Somers' D marginal": orden_msd[:len(orden_iv)],
})

In [ ]:
# somersd_shapley: atribución justa de performance entre varios scores, vía valores de Shapley.
scores_dict = {
    "naive_bayes_woe": proba[:, 1],
    "pipeline_log":    pipe.predict_proba(X_te)[:, 1],
    "piecewise_log":   lr_pw.predict_proba(X_te_pw)[:, 1],
}
screening.somersd_shapley(scores_dict, y_te.values)

---
## **15. ACTUALIZACIÓN INCREMENTAL: `finetune`**

Llega la cosecha del mes siguiente y no querés re-fitear todo desde cero. `finetune` actualiza los
conteos del encoder con los datos nuevos.

`update_prior=False` conserva la tasa base original; `update_prior=True` la recalcula.
En monitoreo de *drift* conviene `False`, para que los WOE sigan siendo comparables contra
la referencia de desarrollo.

In [ ]:
woe_ft = FastWoe(random_state=RANDOM_STATE).fit(X_tr, y_tr)

antes = woe_ft.get_mapping("checking_status")[["category", "count", "event_rate", "woe"]]
woe_ft.finetune(X_te, y_te, update_prior=False)
despues = woe_ft.get_mapping("checking_status")[["category", "count", "event_rate", "woe"]]

comparacion = antes.merge(despues, on="category", suffixes=("_antes", "_despues"))
comparacion["delta_woe"] = comparacion["woe_despues"] - comparacion["woe_antes"]
comparacion

> **Uso práctico: monitoreo de estabilidad.** Fiteás en la ventana de desarrollo, hacés `finetune`
> con la cosecha nueva y mirás `delta_woe`. Un salto grande en un bin es una señal de *drift*
> en esa variable, detectada sin construir un PSI aparte.

---
## **16. MULTICLASS (BREVE)**

Con un target de 3+ clases, FastWoe hace **one-vs-rest**: genera un WOE por clase y por variable.
En scoring crediticio lo habitual es binario, pero sirve para segmentaciones tipo
*al día / mora temprana / mora dura*.

In [ ]:
# Target de 3 clases construido a partir del riesgo estimado, solo para la demo.
p_riesgo = woe.predict_proba(X_tr)[:, 1]
y_multi = pd.Series(
    pd.qcut(p_riesgo, q=3, labels=[0, 1, 2]).astype(int),
    index=X_tr.index, name="riesgo_3c",
)
print(y_multi.value_counts().sort_index().to_dict())

woe_mc = FastWoe(random_state=RANDOM_STATE).fit(X_tr, y_multi)
X_mc = woe_mc.transform(X_tr)

print(f"\n{X_tr.shape[1]} variables  ->  {X_mc.shape[1]} columnas (una por variable y clase)")
print("Nomenclatura:", list(X_mc.columns)[:6])
X_mc.head()

In [ ]:
# Mapeo y predicciones para una clase puntual.
display(woe_mc.get_mapping("duration", class_label=2))

proba_c2 = woe_mc.predict_proba_class(X_tr, class_label=2)
ci_c2    = woe_mc.predict_ci_class(X_tr, class_label=2, alpha=0.05)

print(f"P(clase 2) shape: {proba_c2.shape} | IC shape: {ci_c2.shape}")
print(f"predict_proba (todas las clases): {woe_mc.predict_proba(X_tr).shape}")

---
## **17. CIERRE: EQUIVALENCIAS CON `woe_example/utils.py`**

Traducción directa función por función. El análisis completo —incluidas las diferencias de
convención y las rutas de adopción— está en [`README.md`](README.md).

| Tu `utils.py` | Equivalente en FastWoe |
|---|---|
| `clasificar_variables()` | automático, vía `numerical_threshold` |
| `calcular_metricas_bin()` | interno; se ve en `get_mapping()` |
| `calcular_bivariados()` | `fit()` + `get_mapping()` / `get_feature_stats()` / `get_iv_analysis()` |
| `discretizar_variables()` | `binning_method` + `tree_kwargs` / `binner_kwargs` |
| `percentiles_precalculados` | `fit(train)` → `transform(test)` |
| `calcular_woe_mapping()` | `get_all_mappings()` |
| `aplicar_woe()` | `transform()` |
| `exportar_multiples_bivariados_excel()` | `style_woe_mapping()` / `style_iv_analysis()` |
| *(no existe)* | `woe_se`, IC por bin, `get_iv_analysis()` con significancia |
| *(no existe)* | `monotonic_cst` — monotonicidad forzada |
| *(no existe)* | `assign_pieces()` + logística → el paso a *scorecard* |

**Las dos diferencias de convención**, en una línea cada una:

1. **Signo** — `WOE_utils = −WOE_fastwoe`. Vos usás `ln(%buenos/%malos)`, FastWoe usa `ln(P(malo)/P(bueno))`.
2. **Escala del IV** — `IV_utils ≈ 100 × IV_fastwoe`, porque guardás los porcentajes en puntos porcentuales.

**Lo que NO es una diferencia:** el centrado. Al calcular el WOE sobre `%buenos_total` y `%malos_total`,
tu fórmula **ya está centrada en la tasa poblacional**, igual que la resta explícita del *prior* que hace
FastWoe. Las dos son la misma cosa cambiada de signo:

```
ln( (bᵢ/B) / (mᵢ/M) )  =  −[ ln(mᵢ/bᵢ) − ln(M/B) ]
                            └ log-odds bin ┘ └ prior ┘
```

La verificación numérica de las dos —sobre `checking_status` de este mismo dataset— está en
el [`README.md`](README.md), §4.

In [ ]:
# Cierre: el bin más riesgoso tiene WOE POSITIVO en FastWoe, negativo en tu convención.
m = woe.get_mapping("duration")[["category", "count", "event_rate", "woe"]]
peor = m.loc[m["event_rate"].idxmax()]

print(f"Bin más riesgoso : {peor['category']}")
print(f"  tasa de malos  : {peor['event_rate']:.2%}")
print(f"  WOE (FastWoe)  : {peor['woe']:+.4f}")
print(f"  WOE (tu utils) : {-peor['woe']:+.4f}   <- signo invertido")